*0.2 Math / ML basics*

# UMAP

**The situation.** 5,000 support messages, no labels. Before paying for labelling, the team wants to see what kinds of messages exist — are there 3 topics or 30? PCA to two dimensions gives one grey blob: the clusters are there, but in directions PCA's straight-line view cannot separate.

**UMAP.** Instead of straight lines, it builds a graph of each point's nearest neighbours in the full space and then lays that graph out in two dimensions, keeping neighbours together. Clusters that are tangled in 1,536 dimensions come out as separate islands. It is the standard tool for looking at embeddings.

In [1]:
# Load OPENAI_API_KEY from the .env file. The OpenAI clients read it from the environment.
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())
MODEL = "gpt-4o-mini"

**Embed 90 messages from three topics, squash them, and check that the islands match the topics.** Labels are used only to check, never to fit.

In [2]:
import warnings

import numpy as np
import umap
from openai import OpenAI
from sklearn.cluster import KMeans

warnings.filterwarnings("ignore", module="umap")
from sklearn.metrics import adjusted_rand_score

templates = {
    "billing": ["charged twice for {x}", "refund for {x} not received", "invoice for {x} is wrong"],
    "technical": ["{x} page will not load", "error 500 when opening {x}", "app crashes on {x}"],
    "account": ["reset password for {x}", "change email on {x}", "delete my {x} account"],
}
texts = []
labels = []
for label, patterns in templates.items():
    for pattern in patterns:
        for x in (
            "order",
            "the dashboard",
            "my plan",
            "the mobile app",
            "the report",
            "the team workspace",
            "billing",
            "settings",
            "the export",
            "the API",
        ):
            texts.append(pattern.format(x=x))
            labels.append(label)

client = OpenAI(timeout=60)
vectors = []
for item in client.embeddings.create(model="text-embedding-3-small", input=texts).data:
    vectors.append(item.embedding)
vectors = np.array(vectors, dtype=np.float32)

points = umap.UMAP(n_neighbors=10, min_dist=0.1, random_state=0).fit_transform(
    vectors
)  # 1536 → 2, no labels used
found = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(
    points
)  # find islands in the 2-D picture
agreement = adjusted_rand_score(labels, found)
print("2-D points shape:", points.shape)
print("islands found vs true topics, agreement (1.0 = perfect):", round(agreement, 2))
for topic in templates:
    mask = np.array(labels) == topic
    print(f"{topic:<10} centre x={points[mask, 0].mean():>6.2f}  y={points[mask, 1].mean():>6.2f}")
assert agreement > 0.7

2-D points shape: (90, 2)
islands found vs true topics, agreement (1.0 = perfect): 1.0
billing    centre x=  5.57  y= -1.99
technical  centre x= -4.43  y=  8.75
account    centre x= 13.90  y= 13.79


**Reading the output.** Three islands, and they line up with the three topics almost perfectly — without a single label going into UMAP. That is the picture you would show the team: "there are three kinds of message, here is how many of each."

```
        ○○○○ account                 ●●●● billing
       ○○○○○                        ●●●●●
                     ▲▲▲▲ technical
                    ▲▲▲▲▲
```

**The rule to remember.** UMAP is for *seeing* structure in embeddings: clusters, outliers, duplicates. Not for measuring — distances between islands mean little.

| Use it when | Don't when | Instead use |
|---|---|---|
| exploring unlabelled text; checking that an embedding model separates your categories | the reduced vectors feed a downstream calculation | PCA (honest distances) or the full vectors |

**Watch out**
- Different `random_state` or `n_neighbors` → a different picture. Try a few before believing one.
- Island size and gaps are artefacts of the layout, not of the data.
- Fit on a sample of ~10k points; UMAP on millions takes hours and shows the same thing.